# 🌐 Notebook 3 — Carga de Datos y APIs Públicas
### Unidad 2 | Python para Ciencia de Datos | UADE

---

Este notebook cubre las principales formas de obtener datos para análisis:
1. Cargar archivos locales (CSV, Excel, JSON)
2. Cargar datos desde GitHub y Google Drive
3. Descargar datasets de Kaggle
4. Consumir APIs públicas
5. Directorio de fuentes de datos abiertas


---
## 1. Cargar archivos locales

### CSV — el formato más común en Ciencia de Datos


In [ ]:
import pandas as pd

# Desde archivo local
# df = pd.read_csv("datos/mi_archivo.csv")

# Opciones útiles de read_csv:
# pd.read_csv("archivo.csv",
#     sep=";",           # separador (default: coma)
#     encoding="utf-8",  # codificación de caracteres
#     nrows=1000,        # leer solo las primeras 1000 filas
#     skiprows=2,        # saltar las primeras 2 filas
#     index_col="ID",    # usar columna ID como índice
#     parse_dates=["fecha"]  # parsear columna como fecha
# )

# Ejemplo con datos en línea
url_csv = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_titanic = pd.read_csv(url_csv)
print("Titanic dataset:", df_titanic.shape)
df_titanic.info()

In [ ]:
# Excel
# df = pd.read_excel("archivo.xlsx", sheet_name="Hoja1")

# JSON
# df = pd.read_json("archivo.json")
# df = pd.read_json("archivo.json", orient="records")

# SQL (requiere conexión a base de datos)
# import sqlite3
# conn = sqlite3.connect("database.db")
# df = pd.read_sql("SELECT * FROM tabla", conn)

print("Formatos soportados por pandas:")
for fmt in ["CSV", "Excel (.xlsx)", "JSON", "SQL", "Parquet", "HTML tables", "Clipboard"]:
    print(f"  ✅ {fmt}")

---
## 2. Cargar desde GitHub

GitHub es una excelente fuente de datasets públicos. La clave es usar la URL del archivo **raw** (no la URL de la página web del archivo).

**Cómo obtener la URL raw:**
1. Ir al archivo en GitHub
2. Click en el botón **Raw** 
3. Copiar la URL de la barra del navegador


In [ ]:
import pandas as pd

# URL del archivo RAW en GitHub (no la URL de la página)
url_github = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(url_github)
print(f"Cargado desde GitHub: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head(3)

---
## 3. Cargar desde Google Drive

Para compartir datasets en clase, Google Drive es muy práctico.

### Cómo obtener el link de descarga directa:
1. Subir el archivo a Drive
2. Click derecho → **Compartir** → Cualquiera con el link
3. Copiar el link (tiene formato: `https://drive.google.com/file/d/FILE_ID/view`)
4. Extraer el `FILE_ID` y construir la URL de descarga


In [ ]:
# Función utilitaria para cargar desde Google Drive
def load_from_drive(share_link):
    """
    Convierte un link de Google Drive compartido
    en una URL de descarga directa y carga el CSV.
    """
    file_id = share_link.split('/d/')[1].split('/')[0]
    url = f"https://drive.google.com/uc?id={file_id}"
    return pd.read_csv(url)

# Ejemplo (el mismo dataset de tacos que usamos en el NB3)
link_drive = "https://drive.google.com/file/d/1AALuSA5Dq0-bsIwxUjZdnduH6KUBYOg3/view?usp=sharing"

# df_drive = load_from_drive(link_drive)
# df_drive.head()

print("Función lista para usar con cualquier link de Google Drive ✅")

---
## 4. Directorio de fuentes de datos abiertas 🗂️

| Plataforma | URL | Tipo de datos |
|---|---|---|
| **Kaggle** | https://www.kaggle.com/datasets | Todo tipo, competencias |
| **Hugging Face** | https://huggingface.co/datasets | NLP, imágenes, audio, tabular |
| **UCI ML Repository** | https://archive.ics.uci.edu | Clásicos de ML académico |
| **Makeover Monday** | https://www.makeovermonday.co.uk | Visualización, semanal |
| **Our World in Data** | https://ourworldindata.org | Datos globales, mapas |
| **Google Dataset Search** | https://datasetsearch.research.google.com | Buscador de datasets |
| **Data.gov** | https://data.gov | Datos del gobierno de EE.UU. |
| **datos.gob.ar** | https://datos.gob.ar | Datos abiertos de Argentina |
| **PhysioNet** | https://physionet.org | Datos biomédicos y ECG |
| **World Bank Open Data** | https://data.worldbank.org | Indicadores económicos globales |
| **Awesome Public Datasets** | https://github.com/awesomedata/awesome-public-datasets | Curado por la comunidad |


---
## 5. APIs públicas — ejemplo: API del Banco Mundial

Las APIs permiten acceder a datos actualizados en tiempo real. El Banco Mundial ofrece una API REST gratuita con miles de indicadores económicos para todos los países.

> 📌 La exploración completa de la API del Banco Mundial se verá en clase por separado.  
> Acá mostramos la estructura básica para consumir cualquier API REST con Python.


In [ ]:
import pandas as pd
import requests

# Estructura básica para consumir una API REST
def llamar_api(url, params=None):
    """
    Hace una petición GET a una API y retorna el JSON como diccionario.
    """
    respuesta = requests.get(url, params=params)
    
    if respuesta.status_code == 200:
        return respuesta.json()
    else:
        print(f"Error {respuesta.status_code}: {respuesta.text}")
        return None

# Ejemplo: API del Banco Mundial — GDP per cápita de Argentina
url_wb = "https://api.worldbank.org/v2/country/AR/indicator/NY.GDP.PCAP.CD"
params = {"format": "json", "per_page": 10, "mrv": 10}  # últimos 10 años

datos = llamar_api(url_wb, params)

if datos:
    # La respuesta tiene 2 elementos: [metadata, data]
    registros = datos[1]
    df_gdp = pd.DataFrame([{
        "año":   r["date"],
        "gdp_pc": r["value"],
        "pais":  r["country"]["value"]
    } for r in registros if r["value"] is not None])
    
    df_gdp = df_gdp.sort_values("año")
    print("GDP per cápita — Argentina (USD corrientes)")
    print(df_gdp)

In [ ]:
# Visualizar la serie de tiempo
import matplotlib.pyplot as plt

if 'df_gdp' in dir() and not df_gdp.empty:
    plt.figure(figsize=(10, 4))
    plt.plot(df_gdp["año"], df_gdp["gdp_pc"], marker="o", color="steelblue", linewidth=2)
    plt.title("GDP per cápita — Argentina (Banco Mundial)")
    plt.xlabel("Año")
    plt.ylabel("USD corrientes")
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 🔑 Resumen — Formas de cargar datos

| Fuente | Código |
|---|---|
| Archivo local CSV | `pd.read_csv("archivo.csv")` |
| URL directa | `pd.read_csv("https://...")` |
| GitHub (URL raw) | `pd.read_csv("https://raw.githubusercontent.com/...")` |
| Google Drive | `pd.read_csv(f"https://drive.google.com/uc?id={file_id}")` |
| Excel | `pd.read_excel("archivo.xlsx")` |
| JSON | `pd.read_json("archivo.json")` |
| API REST | `requests.get(url).json()` → `pd.DataFrame(datos)` |
